In [14]:
from sklearn.tree import DecisionTreeClassifier
import pandas as pd
import numpy as np

# ۱. خواندن دیتاست
df = pd.read_csv('../Data/colours_rgb_shades.csv')
# حذف ردیف‌هایی که مقدار خالی دارند
df = df.dropna(subset=['Color Name', 'R;G;B Dec'])

# ۲. جداسازی مقادیر R و G و B از ستون مربوطه
rgb = df['R;G;B Dec'].str.split(';', expand=True).astype(int)
X = rgb
y = df['Color Name']

# ۳. ساخت و آموزش مدل درخت تصمیم
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X, y)

# ۴. تابعی برای تبدیل مدل به کدهای ساده if/else پایتون
def tree_to_code(tree, feature_names, class_names):
    tree_ = tree.tree_
    feature_name = [feature_names[i] if i != -2 else "undefined!" for i in tree_.feature]
    lines = ["def predict_color(r, g, b):"]
    
    def recurse(node, depth):
        indent = "    " * depth
        if tree_.feature[node] != -2:
            name = feature_name[node]
            threshold = tree_.threshold[node]
            lines.append(f"{indent}if {name} <= {threshold}:")
            recurse(tree_.children_left[node], depth + 1)
            lines.append(f"{indent}else:")
            recurse(tree_.children_right[node], depth + 1)
        else:
            class_idx = np.argmax(tree_.value[node][0])
            class_name = str(class_names[class_idx]).replace('"', '\\"')
            lines.append(f"{indent}return \"{class_name}\"")

    recurse(0, 1)
    return "\n".join(lines)

# ۵. تولید و ذخیره کد برای میکروکنترلر
code = tree_to_code(clf, ["r", "g", "b"], clf.classes_)
with open("color_model.py", "w", encoding="utf-8") as f:
    f.write(code)

print("مدل آموزش دید و فایل color_model.py ساخته شد!")

KeyError: ['R;G;B Dec']

In [3]:
import json
import numpy as np

# فرض می‌کنیم مدل clf شما قبلاً آموزش دیده است
tree = clf.tree_

# استخراج ساختار درخت
model_data = {
    "features": tree.feature.tolist(),
    "thresholds": tree.threshold.tolist(),
    "left": tree.children_left.tolist(),
    "right": tree.children_right.tolist(),
    "classes": [str(clf.classes_[np.argmax(v)]) for v in tree.value]
}

# ذخیره به عنوان فایل json
with open("model.json", "w") as f:
    json.dump(model_data, f)

In [4]:
import json

# لود کردن پارامترهای مدل از فایل (فقط یک بار در ابتدای برنامه)
with open('model.json', 'r') as f:
    model = json.load(f)

def predict_color(r, g, b):
    node = 0
    inputs = [r, g, b]
    
    # تا زمانی که به برگ درخت (کلاس نهایی) نرسیده‌ایم (-1 یعنی برگ)
    while model["left"][node] != -1:
        feature_idx = model["features"][node]
        if inputs[feature_idx] <= model["thresholds"][node]:
            node = model["left"][node]
        else:
            node = model["right"][node]
            
    return model["classes"][node]

# استفاده:
print(predict_color(119, 136, 153))

LightSlateGrey


In [ ]:
import json

# لود کردن پارامترهای مدل از فایل (فقط یک بار در ابتدای برنامه)
with open('model.json', 'r') as f:
    model = json.load(f)

def predict_color(r, g, b):
    node = 0
    inputs = [r, g, b]
    
    # تا زمانی که به برگ درخت (کلاس نهایی) نرسیده‌ایم (-1 یعنی برگ)
    while model["left"][node] != -1:
        feature_idx = model["features"][node]
        if inputs[feature_idx] <= model["thresholds"][node]:
            node = model["left"][node]
        else:
            node = model["right"][node]
            
    return model["classes"][node]
print("Reading saved colors from file...\n")

try:
    # باز کردن فایلی که سنسور نوشته است
    with open("saved_colors.txt", "r") as file:
        line_number = 1
        for line in file:
            # پاک کردن فواصل اضافی و خطوط خالی
            line = line.strip()
            if line:
                # جدا کردن اعداد با کاما
                r_str, g_str, b_str = line.split(",")
                r, g, b = int(r_str), int(g_str), int(b_str)
                
                # ارسال به مدل برای پیش‌بینی
                color_name = predict_color(r, g, b)
                
                print(f"Reading {line_number} | RGB: ({r}, {g}, {b}) -> Predicted Color: {color_name}")
                line_number += 1

except OSError:
    print("Error: 'saved_colors.txt' not found! Please run the sensor code first.")

In [5]:
import numpy as np

# فرض می‌کنیم مدل clf آموزش دیده است
tree = clf.tree_

features = tree.feature.tolist()
# گرد کردن اعداد اعشاری برای کاهش حجم فایل
thresholds = [round(x, 2) for x in tree.threshold.tolist()]
left = tree.children_left.tolist()
right = tree.children_right.tolist()

# استخراج تمام کلاس‌ها
node_classes = [str(clf.classes_[np.argmax(v)]) for v in tree.value]

# تکنیک فشرده‌سازی: حذف نام‌های تکراری و استفاده از ایندکس
unique_classes = list(set(node_classes))
class_indices = [unique_classes.index(c) for c in node_classes]

# ذخیره به عنوان یک ماژول پایتون
with open("model_data.py", "w", encoding="utf-8") as f:
    f.write(f"features = {features}\n")
    f.write(f"thresholds = {thresholds}\n")
    f.write(f"left = {left}\n")
    f.write(f"right = {right}\n")
    f.write(f"unique_classes = {unique_classes}\n")
    f.write(f"class_indices = {class_indices}\n")

print("فایل model_data.py با موفقیت و بهینه ساخته شد!")

فایل model_data.py با موفقیت و بهینه ساخته شد!


In [9]:
# =========================
# Main program
# =========================
import gc
import model_data # وارد کردن مدل بهینه شده

# فراخوانی زباله‌روب برای یکپارچه‌سازی RAM قبل از شروع کار
gc.collect()

R, G, B = 255, 136, 255

def predict_color(r, g, b):
    node = 0
    inputs = [r, g, b]
    
    # تا زمانی که به برگ درخت (کلاس نهایی) نرسیده‌ایم (-1 یعنی برگ)
    while model_data.left[node] != -1:
        feature_idx = model_data.features[node]
        if inputs[feature_idx] <= model_data.thresholds[node]:
            node = model_data.left[node]
        else:
            node = model_data.right[node]
            
    # پیدا کردن نام رنگ با استفاده از ایندکس بهینه شده
    idx = model_data.class_indices[node]
    return model_data.unique_classes[idx]

print(f"RGB: ({R}, {G}, {B}) -> Predicted: ", predict_color(R, G, B))



RGB: (255, 136, 255) -> Predicted:  grey53


In [11]:
import numpy as np

tree = clf.tree_

features = tree.feature.tolist()
thresholds = tree.threshold.tolist()
left = tree.children_left.tolist()
right = tree.children_right.tolist()

node_classes = [str(clf.classes_[np.argmax(v)]) for v in tree.value]
unique_classes = list(set(node_classes))
class_indices = [unique_classes.index(c) for c in node_classes]

with open("model_data.py", "w", encoding="utf-8") as f:
    f.write("import array\n\n") # استفاده از آرایه‌های سی
    # 'h' برای اعداد صحیح کوتاه (۲ بایت) و 'f' برای اعشاری (۴ بایت)
    f.write(f"features = array.array('h', {features})\n")
    f.write(f"thresholds = array.array('f', {thresholds})\n")
    f.write(f"left = array.array('h', {left})\n")
    f.write(f"right = array.array('h', {right})\n")
    f.write(f"unique_classes = {unique_classes}\n")
    f.write(f"class_indices = array.array('h', {class_indices})\n")

In [13]:
import pandas as pd

# خواندن دیتاست اصلی
df = pd.read_csv('../Data/colours_rgb_shades.csv').dropna(subset=['Color Name', 'R;G;B Dec'])

# ساخت فایل متنی ساده
with open("palette.txt", "w", encoding="utf-8") as f:
    for index, row in df.iterrows():
        r, g, b = map(int, row['R;G;B Dec'].split(';'))
        name = row['Color Name'].replace('"', '').replace("'", "").strip()
        f.write(f"{r},{g},{b},{name}\n")

print("فایل palette.txt ساخته شد! آن را روی Pyboard کپی کنید.")

فایل palette.txt ساخته شد! آن را روی Pyboard کپی کنید.


In [18]:
from sklearn.tree import DecisionTreeClassifier
import pandas as pd
import numpy as np

# ۱. خواندن دیتاست جدید
df = pd.read_csv('../Data/simple_colors.csv')

# ۲. جداسازی ویژگی‌ها و لیبل‌ها
X = df[['R', 'G', 'B']]
y = df['Color Name']

# ۳. آموزش مدل
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X, y)

# ۴. استخراج اطلاعات درخت برای میکروپایتون
tree = clf.tree_
features = tree.feature.tolist()
thresholds = [round(x, 2) for x in tree.threshold.tolist()]
left = tree.children_left.tolist()
right = tree.children_right.tolist()

node_classes = [str(clf.classes_[np.argmax(v)]) for v in tree.value]
unique_classes = list(set(node_classes))
class_indices = [unique_classes.index(c) for c in node_classes]

# ۵. ذخیره مدل
with open("model_data.py", "w", encoding="utf-8") as f:
    f.write(f"features = {features}\n")
    f.write(f"thresholds = {thresholds}\n")
    f.write(f"left = {left}\n")
    f.write(f"right = {right}\n")
    f.write(f"unique_classes = {unique_classes}\n")
    f.write(f"class_indices = {class_indices}\n")

print("مدل آموزش دید! حالا فایل model_data.py را روی Pyboard کپی کنید.")

مدل آموزش دید! حالا فایل model_data.py را روی Pyboard کپی کنید.
